In [1]:
import refinitiv.data as rd
import pandas as pd
import numpy as np


# Default settings markets

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
markets = ["XPAR", "XAMS", "XBRU", "XMAD", "XNYS", "XNAS"]
ESG_SCORE_THRESH_HOLD = 50
MARKET_VALUE_THRESH_HOLD = 1e9

# STOCK SCREENER 

### ESG Score Screening and Market Value Screening

In [4]:
rd.open_session()

<refinitiv.data.session.Definition object at 0x129ec0710 {name='workspace'}>

In [5]:
mic_codes = ", ".join(markets)
screener_query = (
    "SCREEN("
    "U(IN(Equity(active,public,primary))), "
    f"IN(TR.ExchangeMarketIdCode, {mic_codes}), "
    "TR.TRESGScore > 50"
    ")"
)

In [6]:
df_screener = rd.get_data(screener_query,
                          fields =[
                              "TR.CommonName",
                              "TR.TRESGScore",
                              "TR.CompanyMarketCap(CURN=USD)"
                          ]
)

KeyboardInterrupt: 

In [7]:
df_screener = df_screener[df_screener["Company Market Cap"] >= MARKET_VALUE_THRESH_HOLD]
df_screener["Company Market Cap"].min()


NameError: name 'df_screener' is not defined

In [ ]:
df_screener.head()

,Instrument,Company Common Name,ESG Score,Company Market Cap
0,FN.N,Fabrinet,58.673081,20989447166.790001
1,TGLS.N,Tecnoglass Inc,70.471181,2326609522.16
3,AMA.MC,Amadeus IT Group SA,89.508399,26132188596.786499
4,ZWS.N,Zurn Elkay Water Solutions Corp,69.358455,8441645464.14
5,EDEN.PA,Edenred SE,79.999928,5121362302.25413


# Fetch Data for the selected stocks

### Settings for historical data retrieval

In [ ]:
selected_stocks = df_screener.sort_values(by="Company Market Cap", ascending=False)["Instrument"].tolist()
selected_stocks_by_name = df_screener[["Company Common Name", "Instrument"]]
pd.DataFrame(selected_stocks).to_csv("selected_stocks.csv", index=False)
selected_stocks[:5]


['LLY.N', 'JPM.N', 'XOM.N', 'JNJ.N', 'ASML.AS']

In [ ]:
# Rolling window of 3 months
rolling_window = 4 * 21  # Assuming 21 trading days in a month

In [ ]:
END_DATE = pd.Timestamp.today()
START_DATE = END_DATE - pd.Timedelta(days=rolling_window)


### Fetch close price history for the selected stocks

In [ ]:
df_history = rd.get_history(
    universe=selected_stocks[:300],
    interval='1d',
    fields=['TRDPRC_1'],
    start=START_DATE,
    end=END_DATE
)

/opt/anaconda3/envs/final_project/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


### Upload the Start and End date to the database

In [ ]:
df_history.shape
df_history.head()

TRDPRC_1,LLY.N,JPM.N,XOM.N,JNJ.N,ASML.AS,V.N,MA.N,ABBV.N,PG.N,HD.N,...,BOUY.PA,IP.N,VLTO.N,AMCR.N,DGX.N,PERP.PA,KEY.N,RL.N,PSTG.N,TSN.N
Date,,,,,,,,,,,,,,,,,,,,,
2025-12-04,1014.49,316.1,117.14,202.48,957.3,327.1,542.31,228.71,145.36,351.17,...,43.37,39.13,102.89,41.6,184.18,76.7,19.11,356.97,72.2,56.14
2025-12-05,1010.31,315.04,116.54,201.93,951.6,331.24,545.52,226.08,143.45,354.61,...,43.17,39.06,102.16,41.5,182.51,76.88,19.26,368.42,70.43,56.92
2025-12-08,997.59,315.21,115.98,201.62,963.2,326.84,540.44,223.12,138.34,349.91,...,43.71,38.51,99.62,41.25,181.82,75.22,19.39,356.44,71.04,56.22
2025-12-09,982.22,300.51,118.25,199.96,952.9,326.5,537.55,222.99,139.63,345.27,...,43.7,37.58,98.41,40.55,179.62,73.7,19.98,355.53,70.15,55.91
2025-12-10,993.64,310.11,119.54,206.54,946.0,325.73,538.86,225.18,139.82,351.13,...,43.13,39.12,97.75,41.0,179.51,73.32,20.52,357.7,73.65,57.67


In [ ]:
df_history.isna().sum()

TRDPRC_1
LLY.N      0
JPM.N      0
XOM.N      0
JNJ.N      0
ASML.AS    0
          ..
PERP.PA    0
KEY.N      0
RL.N       0
PSTG.N     0
TSN.N      0
Length: 300, dtype: int64

### Cleaning data

In [ ]:
# Fill missing values with the previous day's price
df_history = df_history.fillna(method='ffill')

/var/folders/d3/4j2xrjxn06v329lnxwrwmzs80000gn/T/ipykernel_50406/4017718853.py:2:FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.


In [ ]:
print(df.dtypes)

In [ ]:
# if Date is a column -> set it as index; if not, just ensure index is datetime
df.index = pd.to_datetime(df.index, errors="coerce")
df = df[~df.index.isna()]
df.head()


,FN.N,TGLS.N,AMA.MC,ZWS.N,EDEN.PA,BWXT.N,AMRC.N,STAG.N,GM.N,BAH.N,TRGP.N,AAT.N,TAL.N,LYB.N,HHH.N,DQ.N,WD.N,KMI.N,HCA.N,APAM.AS
Date,,,,,,,,,,,,,,,,,,,,
2025-12-26,478.32,51.37,<NA>,47.84,<NA>,175.88,30.18,37.24,83.06,85.39,182.9,18.9,11.0,43.25,78.9,31.82,60.97,27.19,477.13,<NA>
2025-12-29,472.08,51.6,62.68,47.77,18.47,175.49,30.07,37.13,82.93,85.12,183.93,19.08,10.99,43.45,79.29,30.21,60.73,27.38,474.02,34.76
2025-12-30,461.93,51.55,62.86,47.32,18.765,174.36,29.41,37.22,82.33,85.15,185.64,19.05,11.0,43.66,80.16,29.64,60.8,27.58,473.26,35.5
2025-12-31,455.28,50.32,62.84,46.49,18.91,172.84,29.29,36.76,81.32,84.36,184.5,18.93,10.91,43.3,79.77,29.5,60.15,27.49,466.86,35.24
2026-01-02,479.42,52.04,62.72,46.85,18.785,181.85,30.67,36.92,80.98,84.89,186.77,18.78,11.49,44.39,78.81,29.66,58.72,27.71,470.39,37.28


# Analyst the stock data

### Change in returns 

In [ ]:
df_ret = np.log(df_history).diff().dropna()
df_ret.head()

TRDPRC_1,LLY.N,JPM.N,XOM.N,JNJ.N,ASML.AS,V.N,MA.N,ABBV.N,PG.N,HD.N,...,BOUY.PA,IP.N,VLTO.N,AMCR.N,DGX.N,PERP.PA,KEY.N,RL.N,PSTG.N,TSN.N
Date,,,,,,,,,,,,,,,,,,,,,
2025-12-05,-0.004129,-0.003359,-0.005135,-0.00272,-0.005972,0.012577,0.005902,-0.011566,-0.013227,0.009748,...,-0.004622,-0.001791,-0.00712,-0.002407,-0.009109,0.002344,0.007819,0.031572,-0.024821,0.013798
2025-12-08,-0.01267,0.000539,-0.004817,-0.001536,0.012116,-0.013372,-0.009356,-0.013179,-0.036272,-0.013343,...,0.012431,-0.014181,-0.025177,-0.006042,-0.003788,-0.021829,0.006727,-0.033058,0.008624,-0.012374
2025-12-09,-0.015527,-0.047758,0.019383,-0.008267,-0.010751,-0.001041,-0.005362,-0.000583,0.009282,-0.013349,...,-0.000229,-0.024446,-0.012221,-0.017115,-0.012174,-0.020414,0.029974,-0.002556,-0.012607,-0.005529
2025-12-10,0.01156,0.031446,0.01085,0.032377,-0.007267,-0.002361,0.002434,0.009773,0.00136,0.01683,...,-0.013129,0.040162,-0.006729,0.011036,-0.000613,-0.005169,0.026668,0.006085,0.048688,0.030994
2025-12-11,0.015717,0.023173,0.0,0.016661,-0.005406,0.0593,0.044481,-0.005343,0.0067,0.017867,...,0.005319,-0.001791,0.009976,0.015729,0.01141,0.028769,0.00825,0.027355,0.029697,0.032247


# Strategy Define

The momentum will choose the top stock with the highest price change over the last month with the positive 60 day returns

- 30% of the funds will be fall within this portfolio
- 70% of the funds will be using adjusted-risk return strategy for making sure the stability of the portfolio

### Momentum Strategy

In [ ]:
# Copy the returns change DataFrame to calculate momentum
df_mom = df_ret.copy()

In [ ]:
# Calculate the  momentum of 20 day and 60 day
mom20 = np.exp(df_mom.rolling(window=20).sum()) - 1
mom60 = np.exp(df_mom.rolling(window=60).sum()) - 1

mom20.head()


TRDPRC_1,LLY.N,JPM.N,XOM.N,JNJ.N,ASML.AS,V.N,MA.N,ABBV.N,PG.N,HD.N,...,BOUY.PA,IP.N,VLTO.N,AMCR.N,DGX.N,PERP.PA,KEY.N,RL.N,PSTG.N,TSN.N
Date,,,,,,,,,,,,,,,,,,,,,
2025-12-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-12-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
